# Introduction to Information Retrieval (IR) and Information Extraction (IE)

This notebook provides an introductory overview of the concepts, definitions, and differences between **Information Retrieval (IR)** and **Information Extraction (IE)**. These two fields are closely related but serve fundamentally different purposes in the pipeline of processing unstructured text.

**Learning Objectives:**
By the end of this notebook, you should be able to:
- Explain the difference between IR and IE and when to use each.
- Describe core concepts like Inverted Index, TF-IDF, and Named Entity Recognition (NER).
- Understand how IR and IE work together in modern AI pipelines (e.g., RAG).
- Run hands-on code examples for both IR and IE tasks.

---

## 1. Information Retrieval (IR)

### What is it?
**Information Retrieval** is the process of obtaining information system resources that are relevant to an information need from a collection of those resources. In simpler terms, IR is about **finding the right documents**.

Imagine you are using a search engine (like Google) or searching through your emails for a specific keyword. You aren't asking the system to "understand" the content — you're asking it to *locate* the documents that likely contain the answer.

### Core Characteristics of IR

| Property | Description |
| :--- | :--- |
| **Input** | A query (keyword, phrase, or natural language question) |
| **Output** | A ranked list of documents or passages |
| **Goal** | High **Recall** (find all relevant docs) + high **Precision** (avoid irrelevant docs) |
| **Example** | Searching for `"Climate Change"` in a database of 1 million scientific papers |

### How IR Works: The Inverted Index
Most IR systems rely on an **Inverted Index**. Instead of scanning every document for a word, the system maintains a map of words → documents where they appear.

```
Term         →  Documents
─────────────────────────
"climate"    →  [doc_3, doc_7, doc_42]
"change"     →  [doc_1, doc_3, doc_19]
"economy"    →  [doc_2, doc_7, doc_88]
```

**Ranking** is then done using algorithms like:
- **TF-IDF** (Term Frequency–Inverse Document Frequency): Rewards documents where the query term is frequent *and* rare across the corpus.
- **BM25**: A probabilistic extension of TF-IDF that handles document length normalization — the current industry standard for sparse retrieval.

### Evaluation Metrics for IR
- **Precision@K**: Of the top K results returned, what fraction are relevant?
- **Recall@K**: Of all relevant documents, what fraction were found in the top K?
- **Mean Average Precision (MAP)**: Averaged precision across multiple queries.
- **NDCG** (Normalized Discounted Cumulative Gain): Accounts for the *position* of relevant results — finding a relevant doc at rank 1 is better than at rank 10.

---

### Hands-On: Building a Simple Inverted Index in Python

Let's build a minimal IR system from scratch to see these concepts in action.

In [20]:
from collections import defaultdict
import math
import re
import nltk 
# --- Sample corpus ---
corpus = {
    "doc1": "Climate change is causing rising sea levels and extreme weather events.",
    "doc2": "Machine learning models are used to predict weather patterns.",
    "doc3": "The economy is affected by climate policy and environmental regulations.",
    "doc4": "Deep learning and neural networks power modern AI systems.",
    "doc5": "Climate models use historical data to forecast future warming trends.",
}

def tokenize(text: str) -> list[str]:
    """Lowercase and split on whitespaces."""
    tokens = nltk.word_tokenize(text.lower())
    # the isalpha() method returns True if all the characters are alphabet letters (a-z).
    tokens = [t for t in tokens if t.isalpha()]
    return tokens

    
# --- Build Inverted Index ---
inverted_index: dict[str, set[str]] = defaultdict(set)
for doc_id, text in corpus.items():
    for token in tokenize(text):
        inverted_index[token].add(doc_id)

print("Inverted index entry for 'climate':", inverted_index.get("climate"))
print("Inverted index entry for 'learning':", inverted_index.get("learning"))

Inverted index entry for 'climate': {'doc3', 'doc1', 'doc5'}
Inverted index entry for 'learning': {'doc4', 'doc2'}


In [21]:
# --- TF-IDF Scoring ---

def tf(term: str, doc_tokens: list[str]) -> float:
    """Term Frequency: how often a term appears in a document."""
    return doc_tokens.count(term) / len(doc_tokens) if doc_tokens else 0.0

def idf(term: str, corpus: dict[str, str]) -> float:
    """Inverse Document Frequency: penalizes terms that appear in many documents."""
    n_docs_with_term = sum(1 for text in corpus.values() if term in tokenize(text))
    return math.log((len(corpus) + 1) / (n_docs_with_term + 1)) + 1  # smoothed

def tfidf_search(query: str, corpus: dict[str, str], top_k: int = 3) -> list[tuple[str, float]]:
    """Return top-K documents ranked by TF-IDF score for the given query."""
    query_terms = tokenize(query)
    scores: dict[str, float] = {}

    for doc_id, text in corpus.items():
        doc_tokens = tokenize(text)
        score = sum(tf(term, doc_tokens) * idf(term, corpus) for term in query_terms)
        scores[doc_id] = score

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

    

# --- Run a query ---
query = "climate change models"
results = tfidf_search(query, corpus)

print(f"Query: '{query}'\n")
print(f"{'Rank':<6} {'Doc ID':<8} {'Score':<10} Text")
print("-" * 70)
for rank, (doc_id, score) in enumerate(results, start=1):
    print(f"{rank:<6} {doc_id:<8} {score:<10.4f} {corpus[doc_id]}")

Query: 'climate change models'

Rank   Doc ID   Score      Text
----------------------------------------------------------------------
1      doc1     0.3186     Climate change is causing rising sea levels and extreme weather events.
2      doc5     0.3099     Climate models use historical data to forecast future warming trends.
3      doc2     0.1881     Machine learning models are used to predict weather patterns.


---

## 2. Information Extraction (IE)

### What is it?
**Information Extraction** is the process of automatically extracting structured information from unstructured or semi-structured text. While IR *finds* the document, IE **extracts the specific facts** from within it.

IE transforms raw **text** into **structured data** (tables, database entries, JSON objects).

**Example:**

```
Input text:  "Apple Inc. was founded by Steve Jobs in Cupertino in 1976."

IE output:   {
               "organization": "Apple Inc.",
               "founder": "Steve Jobs",
               "location": "Cupertino",
               "year_founded": 1976
             }
```

### Core IE Tasks

| Task | Description | Example |
| :--- | :--- | :--- |
| **Named Entity Recognition (NER)** | Identify and classify named entities | `"Apple Inc."` → `ORG` |
| **Relation Extraction (RE)** | Identify relationships between entities | `Apple` → `located_in` → `Cupertino` |
| **Event Extraction** | Detect events, participants, time, place | `"Company X acquired Y on Jan 1st"` |
| **Coreference Resolution** | Link mentions referring to the same entity | `"Steve Jobs ... he ..."` → same person |
| **Template Filling** | Populate a predefined schema from text | Fill a job posting schema from a PDF |

### The Evolution of IE Approaches

```
Rule-Based (Regex, Gazetteers)
        ↓  High precision, low recall, brittle
Statistical ML (CRF, HMM, SVM)
        ↓  Requires labeled data, more generalizable
Deep Learning (BiLSTM-CRF, BERT)
        ↓  Context-aware, handles ambiguity ("Apple" fruit vs. company)
LLMs + Prompting (GPT-4, Claude)
        ↓  Few-shot or zero-shot, complex schemas, no retraining needed
```

---

### Hands-On: Named Entity Recognition with spaCy

Let's use the `spaCy` library to run NER on sample text.

In [1]:
import spacy

# Load the small English model
nlp = spacy.load("en_core_web_sm")

text = (
    "Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne "
    "in Cupertino, California in April 1976. "
    "In 2023, Apple reported revenues of $394 billion."
)

doc = nlp(text)

print(f"{'Entity':<30} {'Label':<15} {'Explanation'}")
print("-" * 70)
for ent in doc.ents:
    print(f"{ent.text:<30} {ent.label_:<15} {spacy.explain(ent.label_)}")

Entity                         Label           Explanation
----------------------------------------------------------------------
Apple Inc.                     ORG             Companies, agencies, institutions, etc.
Steve Jobs                     PERSON          People, including fictional
Steve Wozniak                  PERSON          People, including fictional
Ronald Wayne                   PERSON          People, including fictional
Cupertino                      GPE             Countries, cities, states
California                     GPE             Countries, cities, states
April 1976                     DATE            Absolute or relative dates or periods
2023                           DATE            Absolute or relative dates or periods
Apple                          ORG             Companies, agencies, institutions, etc.
$394 billion                   MONEY           Monetary values, including unit


In [23]:
# Visualize entities inline (works in Jupyter)
from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

### Hands-On: Create Structured Output

In [25]:
from rich import print as rprint
from ie_course.utils import ents_to_dict, print_ents
# Group entities by label
rprint(ents_to_dict(doc))

{
    'ORG': ['Apple Inc.', 'Apple'],
    'PERSON': ['Steve Jobs', 'Steve Wozniak', 'Ronald Wayne'],
    'GPE': ['Cupertino', 'California'],
    'DATE': ['April 1976', '2023'],
    'MONEY': ['$394 billion']
}

## 3. The Synergy: IR + IE in Real Pipelines

In real-world applications, IR and IE are almost always used **together**:

```
User Query
    │
    ▼
┌─────────────────────┐
│   IR System         │  ← Finds the top-K relevant documents
│ (BM25 / Vector DB)  │
└─────────────────────┘
    │  Relevant Documents
    ▼
┌─────────────────────┐
│   IE System / LLM   │  ← Extracts the specific answer from those documents
│ (NER / QA Model)    │
└─────────────────────┘
    │
    ▼
Structured Answer
```

### Motivation: RAG (Retrieval-Augmented Generation)

Modern AI assistants use a pattern called **RAG**, which makes this pipeline concrete:

1. **Retrieval (IR step):** When you ask a question, the system uses *dense vector search* (embedding similarity) or BM25 to find the most relevant chunks of text from a knowledge base.
2. **Generation with Extraction (IE step):** The retrieved text is fed into an LLM as context. The LLM performs on-the-fly IE — reading the passage and extracting the answer in natural language.

**Why RAG beats pure LLMs:** LLMs have a fixed knowledge cutoff; RAG keeps answers grounded in up-to-date documents and reduces hallucination.

---